# Minimum Silver Table 3: Google Shots & Shared Tracing
**Course:** TU Delft DSAIT4000 (Data Management & Engineering) — Assignment 1  
**Target Table:** `silver/google_qec/shot.parquet`  
**Shared Provenance:** `results/part1/source_trace.parquet`

---

### Objectives & Contracts
1. **Schema Fidelity:** Build `silver/google_qec/shot.parquet` where **one row represents one aligned hardware shot** (250,000 rows across 5 experiments).
2. **Companion-File Alignment & Validation:**
   - Pre-validate that all companion binary and text files match expected byte strides and shot counts before publishing.
   - Align 8 companion sources per shot:
     - `measurements.b8`: Packed Stim binary measurements.
     - `detection_events.b8`: Packed Stim binary detector checks.
     - `sweep.b8`: Packed sweep bits.
     - `obs_flips_actual.01`: Ground-truth logical observable flip label ($y$).
     - 4 baseline decoder prediction files (`.01`): Belief Matching, Correlated Matching, PyMatching, and Tensor Network Contraction.
3. **Strict Column Types:**
   - `source_record_id`: `string` (stable identifier for the aligned shot)
   - `experiment_id`: `string` (foreign key to `silver/google_qec/experiment.parquet`)
   - `shot_index`: `int64` (zero-based shot index $0 \dots 49,999$)
   - `measurement_bits`: `binary` (packed Stim `b8` row)
   - `sweep_bits`: `binary` (packed sweep row)
   - `detector_bits`: `binary` (packed Stim `b8` detector row)
   - `detector_event_count`: `int32` (number of set detector bits)
   - `actual_observable_flip`: `bool` (actual logical outcome)
   - `belief_matching_prediction`: `bool` (supplied decoder prediction)
   - `correlated_matching_prediction`: `bool` (supplied decoder prediction)
   - `pymatching_prediction`: `bool` (supplied decoder prediction)
   - `tensor_network_contraction_prediction`: `bool` (supplied decoder prediction)
4. **QEC Conceptual Separation:**
   - Keep measurements, detector events, actual outcomes, predictions, and decoder mistakes strictly separated.
5. **Shared Provenance Rule:**
   - Record companion-file alignment in `results/part1/source_trace.parquet` using the modular `save_source_traces` helper.
6. **Dual-Lake Persistence:** Write Parquet locally and synchronize to MinIO bucket `quantum-lake`.


In [1]:
import hashlib
import io
import os
from pathlib import Path
import zipfile
import yaml

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Platform helpers from starter package
from quantum_lake_student.config import Settings
from quantum_lake_student.connections import minio_client
from quantum_lake_student.formats import b8_record_bytes, parse_01_records
from quantum_lake_student.tracing import save_source_traces

# Detect if running in container (/workspace) or locally
BASE_DIR = Path("/workspace") if Path("/workspace").exists() else Path(".").resolve()
print(f"Base Directory: {BASE_DIR}")

# Load configuration
settings = Settings.from_environment()
print(f"Lake Backend: {settings.lake_backend}")
print(f"MinIO Endpoint: {settings.s3_endpoint} (Bucket: {settings.s3_bucket})")


Base Directory: /workspace
Lake Backend: minio
MinIO Endpoint: http://minio:9000 (Bucket: quantum-lake)


### Step 1: Environment & Platform Setup Details

1. **Library Imports & Roles:**
   - `b8_record_bytes`: Computes exact byte strides for little-endian packed binary arrays (e.g. $\lceil 200/8 \rceil = 25$ bytes, $\lceil 209/8 \rceil = 27$ bytes).
   - `parse_01_records`: Robustly parses newline-delimited ASCII zero/one target files into integer lists.
   - `save_source_traces`: The modular helper from `quantum_lake_student.tracing` ensuring idempotent provenance updates.
   - `pyarrow`: Creates memory-efficient Arrow tables and serializes Parquet using ZSTD compression.
2. **Environment & Path Detection:**
   - Automatically handles container paths (`/workspace`) versus local working directories.
   - Connects to MinIO using `Settings.from_environment()`.


In [2]:
# Locate Google Bronze zip object (try MinIO first, fallback to local path)
bronze_object_name = "bronze/source=google_qec/google-surface-code-curated.zip"
bronze_bytes = None

if settings.lake_backend == "minio":
    try:
        client = minio_client(settings)
        print(f"Fetching '{bronze_object_name}' from MinIO...")
        response = client.get_object(settings.s3_bucket, bronze_object_name)
        bronze_bytes = response.read()
        response.close()
        response.release_conn()
        print(f"Successfully retrieved from MinIO ({len(bronze_bytes):,} bytes)")
    except Exception as e:
        print(f"MinIO fetch warning: {e}. Falling back to local file.")

if bronze_bytes is None:
    # Try common local mounts
    candidates = [
        Path("/course-data/raw/source=google_qec/google-surface-code-curated.zip"),
        BASE_DIR.parent / "datasets/student-bundle/core/raw/source=google_qec/google-surface-code-curated.zip",
        Path("datasets/student-bundle/core/raw/source=google_qec/google-surface-code-curated.zip"),
    ]
    for p in candidates:
        if p.exists():
            print(f"Reading from local path: {p}")
            bronze_bytes = p.read_bytes()
            break

assert bronze_bytes is not None, "Could not locate Bronze Google QEC archive!"

# Compute Bronze SHA-256 hash (Required for source_trace.parquet)
bronze_sha256 = hashlib.sha256(bronze_bytes).hexdigest()
print(f"Google Bronze Archive SHA-256: {bronze_sha256}")
print(f"Google Bronze Archive Size:    {len(bronze_bytes):,} bytes")


Fetching 'bronze/source=google_qec/google-surface-code-curated.zip' from MinIO...
Successfully retrieved from MinIO (14,638,673 bytes)
Google Bronze Archive SHA-256: 5d6a24f89f055883a49910979490be4baef54d28bf9a0f8e096a1d6c46d1ea56
Google Bronze Archive Size:    14,638,673 bytes


### Step 2: Bronze Ingestion & Cryptographic Checksum Details

1. **In-Memory Streaming Without Full Disk Extraction:**
   - Reads the curated 14.6 MB Google archive into memory without extracting thousands of loose files to the filesystem.
2. **Audit Hash Verification (`input_sha256`):**
   - Computes the SHA-256 checksum (`5d6a24f89f055883a49910979490be4baef54d28bf9a0f8e096a1d6c46d1ea56`).
   - Every shot record generated from this archive points to this verified checksum in `results/part1/source_trace.parquet`.


In [3]:
source_record_ids = []
experiment_ids = []
shot_indices = []
measurement_bits_list = []
sweep_bits_list = []
detector_bits_list = []
detector_event_counts = []
actual_flips_list = []
bm_preds = []
cm_preds = []
pym_preds = []
tnc_preds = []

trace_records = []

decoder_keys = [
    ("obs_flips_predicted_by_belief_matching.01", bm_preds),
    ("obs_flips_predicted_by_correlated_matching.01", cm_preds),
    ("obs_flips_predicted_by_pymatching.01", pym_preds),
    ("obs_flips_predicted_by_tensor_network_contraction.01", tnc_preds),
]

with zipfile.ZipFile(io.BytesIO(bronze_bytes)) as z:
    exp_dirs = sorted(list(set(name.split("/")[0] for name in z.namelist() if "/" in name)))
    print(f"Processing {len(exp_dirs)} experiments (50,000 shots each):")

    for exp in exp_dirs:
        # 1. Load experiment parameters from properties.yml
        prop = yaml.safe_load(z.read(f"{exp}/properties.yml").decode("utf-8"))
        shots = int(prop["shots"])
        meas_stride = b8_record_bytes(int(prop["circuit_measurements"]))
        det_stride = b8_record_bytes(int(prop["circuit_detectors"]))
        sweep_stride = b8_record_bytes(int(prop["circuit_sweep_bits"]))

        # 2. Read companion raw binary streams
        meas_raw = z.read(f"{exp}/measurements.b8")
        det_raw = z.read(f"{exp}/detection_events.b8")
        has_sweep = f"{exp}/sweep.b8" in z.namelist()
        sweep_raw = z.read(f"{exp}/sweep.b8") if has_sweep else b""

        # 3. Read companion ASCII zero/one target and prediction streams
        act_raw = parse_01_records(z.read(f"{exp}/obs_flips_actual.01"))
        dec_streams = {}
        for fname, target_list in decoder_keys:
            dec_streams[fname] = parse_01_records(z.read(f"{exp}/{fname}"))

        # 4. Rigorous Companion-File Length Pre-Validation
        assert len(meas_raw) == shots * meas_stride, f"Measurement length mismatch in {exp}"
        assert len(det_raw) == shots * det_stride, f"Detector length mismatch in {exp}"
        if has_sweep:
            assert len(sweep_raw) == shots * sweep_stride, f"Sweep length mismatch in {exp}"
        assert len(act_raw) == shots, f"Actual flip count mismatch in {exp}"
        for fname, dstream in dec_streams.items():
            assert len(dstream) == shots, f"Decoder prediction count mismatch in {exp}/{fname}"

        # 5. Assemble aligned shot records
        for i in range(shots):
            shot_id = f"google_qec:{exp}:shot:{i}"
            
            # Slice fixed-width packed bytes for this shot
            m_chunk = meas_raw[i * meas_stride : (i + 1) * meas_stride]
            d_chunk = det_raw[i * det_stride : (i + 1) * det_stride]
            sw_chunk = sweep_raw[i * sweep_stride : (i + 1) * sweep_stride] if has_sweep else b""
            
            # Count set bits in detector_bits
            event_cnt = int.from_bytes(d_chunk, "little").bit_count()

            # Append to columnar arrays
            source_record_ids.append(shot_id)
            experiment_ids.append(exp)
            shot_indices.append(i)
            measurement_bits_list.append(m_chunk)
            sweep_bits_list.append(sw_chunk)
            detector_bits_list.append(d_chunk)
            detector_event_counts.append(event_cnt)
            actual_flips_list.append(bool(act_raw[i]))

            bm_preds.append(bool(dec_streams["obs_flips_predicted_by_belief_matching.01"][i]))
            cm_preds.append(bool(dec_streams["obs_flips_predicted_by_correlated_matching.01"][i]))
            pym_preds.append(bool(dec_streams["obs_flips_predicted_by_pymatching.01"][i]))
            tnc_preds.append(bool(dec_streams["obs_flips_predicted_by_tensor_network_contraction.01"][i]))

            # Append primary shot trace (links shot to its detector stream in Bronze)
            trace_records.append({
                "source_record_id": shot_id,
                "source_name": "google_qec",
                "bronze_object": bronze_object_name,
                "archive_member": f"{exp}/detection_events.b8",
                "record_locator": f"shot:{i}",
                "input_sha256": bronze_sha256,
            })

        print(f"  ✓ {exp}: 50,000 shots aligned across all 8 companion members")

print(f"\nTotal Silver shot records assembled: {len(source_record_ids):,}")


Processing 5 experiments (50,000 shots each):
  ✓ surface_code_bX_d3_r25_center_3_5: 50,000 shots aligned across all 8 companion members
  ✓ surface_code_bX_d3_r25_center_5_3: 50,000 shots aligned across all 8 companion members


  ✓ surface_code_bX_d3_r25_center_5_7: 50,000 shots aligned across all 8 companion members
  ✓ surface_code_bX_d3_r25_center_7_5: 50,000 shots aligned across all 8 companion members
  ✓ surface_code_bX_d5_r25_center_5_5: 50,000 shots aligned across all 8 companion members

Total Silver shot records assembled: 250,000


### Step 3: Companion File Validation & Shot Assembly Details

1. **Pre-Validation of Companion Streams:**
   - Fulfills the contract instruction: *"Validate all companion-file lengths before publishing the row."*
   - Verifies that `detection_events.b8`, `measurements.b8`, `sweep.b8`, `obs_flips_actual.01`, and all 4 decoder prediction files align shot-for-shot to exactly 50,000 records.
2. **Binary Stride Slicing:**
   - Extracts fixed-width byte blocks without parsing individual bits into nested lists, keeping memory overhead minimal:
     - $d=3$: 27 bytes for measurements (209 bits), 25 bytes for detectors (200 bits), 2 bytes for sweep (9 bits).
     - $d=5$: 79 bytes for measurements (625 bits), 75 bytes for detectors (600 bits), 4 bytes for sweep (25 bits).
3. **Hardware Detector Event Counting:**
   - Computes `detector_event_count` using Python's high-speed `int.from_bytes(d_chunk, 'little').bit_count()`.
   - Because $200$ and $600$ are exact multiples of 8, there are 0 padding bits in `detector_bits`, ensuring the bit count is 100% physically accurate.
4. **Separation of Concepts:**
   - `actual_observable_flip`, decoder predictions, and detector counts remain separate boolean and integer columns. No decoder error flags are computed or conflated here.
5. **Deterministic Shot Lineage:**
   - Assigns `source_record_id = f"google_qec:{exp}:shot:{i}"` linking each shot back to its physical offset in Bronze.


In [4]:
google_shot_schema = pa.schema([
    ("source_record_id", pa.string()),
    ("experiment_id", pa.string()),
    ("shot_index", pa.int64()),
    ("measurement_bits", pa.binary()),
    ("sweep_bits", pa.binary()),
    ("detector_bits", pa.binary()),
    ("detector_event_count", pa.int32()),
    ("actual_observable_flip", pa.bool_()),
    ("belief_matching_prediction", pa.bool_()),
    ("correlated_matching_prediction", pa.bool_()),
    ("pymatching_prediction", pa.bool_()),
    ("tensor_network_contraction_prediction", pa.bool_()),
])

table_shot = pa.Table.from_arrays(
    [
        pa.array(source_record_ids, type=pa.string()),
        pa.array(experiment_ids, type=pa.string()),
        pa.array(shot_indices, type=pa.int64()),
        pa.array(measurement_bits_list, type=pa.binary()),
        pa.array(sweep_bits_list, type=pa.binary()),
        pa.array(detector_bits_list, type=pa.binary()),
        pa.array(detector_event_counts, type=pa.int32()),
        pa.array(actual_flips_list, type=pa.bool_()),
        pa.array(bm_preds, type=pa.bool_()),
        pa.array(cm_preds, type=pa.bool_()),
        pa.array(pym_preds, type=pa.bool_()),
        pa.array(tnc_preds, type=pa.bool_()),
    ],
    schema=google_shot_schema,
)

print("=== Google Shot Table ===")
print(f"Rows: {table_shot.num_rows:,}, Columns: {table_shot.num_columns}")
print(table_shot.schema)


=== Google Shot Table ===
Rows: 250,000, Columns: 12
source_record_id: string
experiment_id: string
shot_index: int64
measurement_bits: binary
sweep_bits: binary
detector_bits: binary
detector_event_count: int32
actual_observable_flip: bool
belief_matching_prediction: bool
correlated_matching_prediction: bool
pymatching_prediction: bool
tensor_network_contraction_prediction: bool


### Step 4: Strict PyArrow Schema & Arrow Table Construction Details

1. **Zero-Copy Columnar Assembly:**
   - Constructs the PyArrow Table directly from pre-typed contiguous columnar arrays (`pa.array(...)`), bypassing intermediate Pandas DataFrame conversions to optimize memory.
2. **Schema Compliance:**
   - Conforms strictly to `silver-tables.md`:
     - Packed byte rows (`measurement_bits`, `sweep_bits`, `detector_bits`) are typed as `pa.binary()`.
     - `detector_event_count` is typed as `pa.int32()`.
     - Labels and decoder predictions are typed as `pa.bool_()`.
     - `shot_index` is typed as `pa.int64()`.
3. **No Index Overhead:**
   - The resulting Arrow table contains strictly the 12 required data columns without artificial index keys.


In [5]:
# Define paths relative to base workspace directory
silver_dir = BASE_DIR / "silver/google_qec"
silver_dir.mkdir(parents=True, exist_ok=True)
silver_parquet_path = silver_dir / "shot.parquet"

results_dir = BASE_DIR / "results/part1"
results_dir.mkdir(parents=True, exist_ok=True)
trace_parquet_path = results_dir / "source_trace.parquet"

# Write Silver table with ZSTD compression
pq.write_table(table_shot, silver_parquet_path, compression="zstd")
print(f"✓ Wrote Silver shot table to: {silver_parquet_path} ({silver_parquet_path.stat().st_size:,} bytes)")

# Upload Silver table to MinIO
if settings.lake_backend == "minio":
    try:
        client = minio_client(settings)
        minio_silver_key = "silver/google_qec/shot.parquet"
        client.fput_object(settings.s3_bucket, minio_silver_key, str(silver_parquet_path))
        print(f"✓ Uploaded Silver shot table to MinIO: {settings.s3_bucket}/{minio_silver_key}")
    except Exception as e:
        print(f"Warning: MinIO upload failed: {e}")

# Save source trace records idempotently using the modular helper
total_traces = save_source_traces(
    new_records=trace_records,
    source_name="google_qec",
    trace_file_path=trace_parquet_path,
    settings=settings,
)
print(f"✓ Master source_trace table updated! Total rows now: {total_traces:,}")


✓ Wrote Silver shot table to: /workspace/silver/google_qec/shot.parquet (16,150,379 bytes)
✓ Uploaded Silver shot table to MinIO: quantum-lake/silver/google_qec/shot.parquet


✓ Master source_trace table updated! Total rows now: 325,603


### Step 5: Atomic Parquet Export & Idempotent Lineage Recording Details

1. **Parquet Compression:**
   - Serializes the 250,000 shots using Zstandard (`zstd`), storing all packed binary measurements, detectors, and decoder predictions in ~16.1 MB.
2. **Object Store Synchronization:**
   - Pushes `shot.parquet` to MinIO bucket `quantum-lake` under `silver/google_qec/shot.parquet`.
3. **Modular Tracing Coexistence:**
   - Calls `save_source_traces` to register the 250,000 Google shot traces into `results/part1/source_trace.parquet`.
   - Preserves previous entries from `qec_syndromes` (75,598 rows) and Google experiment metadata (5 rows).
   - Re-running this cell cleanly refreshes the trace records without generating duplicate rows.


In [6]:
# Read back the written parquet table to guarantee write integrity
verified_table = pq.read_table(silver_parquet_path)
df_check = verified_table.to_pandas()

# 1. Total row count reconciliation
assert len(df_check) == 250000, f"Expected 250,000 shots, got {len(df_check):,}"
assert df_check["source_record_id"].is_unique, "source_record_id must be unique across all shots"

# 2. Per-experiment shot counts and index ranges
exp_counts = df_check["experiment_id"].value_counts().to_dict()
assert len(exp_counts) == 5, f"Expected 5 experiments, got {len(exp_counts)}"
for exp, count in exp_counts.items():
    assert count == 50000, f"Experiment {exp} has {count} shots, expected 50,000"
    exp_shots = df_check[df_check["experiment_id"] == exp]["shot_index"]
    assert exp_shots.min() == 0 and exp_shots.max() == 49999, f"Shot index bounds violated in {exp}"

# 3. Non-nullness and domain checks
assert df_check.isna().sum().sum() == 0, "No null values permitted in Silver"
assert (df_check["detector_event_count"] >= 0).all(), "Detector event counts must be non-negative"

# 4. Verify packed byte lengths by distance
for exp, group in df_check.groupby("experiment_id"):
    is_d5 = "d5" in exp
    expected_det_len = 75 if is_d5 else 25
    expected_meas_len = 79 if is_d5 else 27
    expected_sweep_len = 4 if is_d5 else 2
    
    assert (group["detector_bits"].str.len() == expected_det_len).all()
    assert (group["measurement_bits"].str.len() == expected_meas_len).all()
    assert (group["sweep_bits"].str.len() == expected_sweep_len).all()

# 5. Master trace table reconciliation
df_trace = pq.read_table(trace_parquet_path).to_pandas()
source_counts = df_trace["source_name"].value_counts().to_dict()
assert source_counts.get("qec_syndromes") == 75598, "qec_syndromes traces must equal 75,598"
assert source_counts.get("google_qec") >= 250005, "google_qec traces must cover experiments + shots"

print("✓ All 6 Data Quality Invariants Passed!")
print(f"  • Total Shots:     {len(df_check):,} (5 experiments x 50,000 shots)")
print(f"  • Nulls:           {df_check.isna().sum().sum()}")
print(f"  • Master Traces:   {len(df_trace):,} rows")
print("\n=== Sample Google Shot Records ===")
print(df_check[["source_record_id", "experiment_id", "shot_index", "detector_event_count", "actual_observable_flip", "pymatching_prediction"]].head(5).to_string(index=False))


✓ All 6 Data Quality Invariants Passed!
  • Total Shots:     250,000 (5 experiments x 50,000 shots)
  • Nulls:           0
  • Master Traces:   325,603 rows

=== Sample Google Shot Records ===
                                   source_record_id                     experiment_id  shot_index  detector_event_count  actual_observable_flip  pymatching_prediction
google_qec:surface_code_bX_d3_r25_center_3_5:shot:0 surface_code_bX_d3_r25_center_3_5           0                    27                    True                   True
google_qec:surface_code_bX_d3_r25_center_3_5:shot:1 surface_code_bX_d3_r25_center_3_5           1                    36                    True                  False
google_qec:surface_code_bX_d3_r25_center_3_5:shot:2 surface_code_bX_d3_r25_center_3_5           2                    32                    True                  False
google_qec:surface_code_bX_d3_r25_center_3_5:shot:3 surface_code_bX_d3_r25_center_3_5           3                    25                   F

### Step 6: Post-Build Verification & Integrity Guarantees Details

1. **Reconciliation Verification:**
   - Exactly **250,000 shots** verified across the 5 experiments (50,000 shots each).
   - Zero missing/null values across all columns.
   - Primary key uniqueness across all 250,000 rows.
2. **Byte Length Invariants:**
   - Validates that $d=3$ experiments have strictly 25-byte detector rows and 27-byte measurement rows.
   - Validates that $d=5$ experiments have strictly 75-byte detector rows and 79-byte measurement rows.
3. **Audit Lineage Integrity:**
   - Confirms that `results/part1/source_trace.parquet` maintains complete trace coverage across both `qec_syndromes` (75,598 rows) and `google_qec` (250,005 rows) without record corruption.
